In [4]:
import urllib.request
import urllib.parse
import json
import csv
import time
import pandas as pd

def reconstruct_abstract(inverted_index):
    """
    Reconstructs full text abstract from OpenAlex's abstract_inverted_index.
    OpenAlex stores abstracts as a dict: {'word': [pos1, pos2, ...]}
    """
    if not inverted_index:
        return ""
    position_word_map = {}
    for word, positions in inverted_index.items():
        for pos in positions:
            position_word_map[pos] = word
    sorted_positions = sorted(position_word_map.keys())
    return " ".join([position_word_map[pos] for pos in sorted_positions])

def fetch_openalex_catalysis_abstracts(target_count=1000, from_year=2000, 
                                       output_csv=None, output_json=None, output_excel=None):
    # Auto-generate dynamic names only if custom ones aren't provided
    output_csv = output_csv or f"openalex_catalysts_{target_count}.csv"
    output_json = output_json or f"openalex_catalysts_{target_count}.json"
    output_excel = output_excel or f"openalex_catalysts_{target_count}.xlsx"
    """
    Fetches target_count scientific papers published after from_year on energy catalysis and critical mineral substitution
    from the OpenAlex API, extracts metadata, reconstructs abstracts, and saves simultaneously
    to CSV, JSON, and Excel (.xlsx) formats.
    """
    base_url = "https://api.openalex.org/works"
    
    # Query tailored to catalysis advances for energy efficiency and critical mineral replacement
    search_query = '("catalyst" OR "catalysis" OR "catalytic") AND ("critical mineral" OR "PGM-free" OR "platinum replacement" OR "non-precious" OR "earth-abundant" OR "non-noble") AND ("hydrogen" OR "fuel cell" OR "electrochemical" OR "energy efficiency" OR "non-thermal" OR "sustainable")'
    
    # Polite pool mailto identifier per OpenAlex guidelines
    mailto = "rmiranda@montgomerycollege.edu"
    
    records = []
    cursor = "*"
    per_page = 200  # Maximum page size in OpenAlex REST API
    
    print(f"Starting API fetch for {target_count} records from OpenAlex...")
    
    while len(records) < target_count and cursor:
        params = urllib.parse.urlencode({
            'search': search_query,
            'per_page': per_page,
            'cursor': cursor,
            'mailto': mailto,
            'filter' : f'has_abstract:true,language:en,from_publication_date:{from_year}-01-01'
        })
        url = f"{base_url}?{params}"
        
        try:
            req = urllib.request.Request(url, headers={'User-Agent': f'CMSC299Capstone/1.0 (mailto:{mailto})'})
            with urllib.request.urlopen(req) as response:
                data = json.loads(response.read().decode('utf-8'))
                results = data.get('results', [])
                
                if not results:
                    print("No more results returned by API.")
                    break
                
                for work in results:
               
                    # Check publication year >= from_year
                    pub_year = work.get('publication_year')
                    if pub_year and pub_year < from_year:
                        continue   # Skip papers publisehd before min_year
                
                    # Extract and reconstruct abstract
                    inv_index = work.get('abstract_inverted_index')
                    abstract_text = reconstruct_abstract(inv_index) if inv_index else ""
                    
                    if not abstract_text or len(abstract_text.split()) < 30:
                        continue  # Skip records without substantial abstract text
                    
                    # Extract Authors, Institutions, and Countries
                    authors = []
                    institutions = []
                    countries = []
                    for assoc in work.get('authorships', []):
                        author_name = assoc.get('author', {}).get('display_name')
                        if author_name:
                            authors.append(author_name)
                        for inst in assoc.get('institutions', []):
                            inst_name = inst.get('display_name')
                            country_code = inst.get('country_code')
                            if inst_name and inst_name not in institutions:
                                institutions.append(inst_name)
                            if country_code and country_code not in countries:
                                countries.append(country_code)
                    
                    # Extract OpenAlex Concepts (Knowledge Items / Keywords)
                    concepts = [
                        c.get('display_name') for c in work.get('concepts', [])
                        if c.get('score', 0) > 0.3 and c.get('display_name')
                    ]
                    
                    # Primary Venue / Journal Name
                    primary_loc = work.get('primary_location') or {}
                    source = primary_loc.get('source') or {}
                    journal_name = source.get('display_name', '')
                    
                    record = {
                        'openalex_id': work.get('id', ''),
                        'doi': work.get('doi', ''),
                        'title': work.get('title', ''),
                        'publication_year': work.get('publication_year', ''),
                        'publication_date': work.get('publication_date', ''),
                        'journal': journal_name,
                        'abstract': abstract_text,
                        'authors': "; ".join(authors),
                        'institutions': "; ".join(institutions),
                        'countries': "; ".join(countries),
                        'concepts': "; ".join(concepts),
                        'cited_by_count': work.get('cited_by_count', 0)
                    }
                    
                    records.append(record)
                    if len(records) >= target_count:
                        break
                
                cursor = data.get('meta', {}).get('next_cursor')
                print(f"Fetched {len(records)} / {target_count} valid records...")
                time.sleep(0.2)  # Respect API rate limits
                
        except Exception as e:
            print(f"Error querying OpenAlex API: {e}")
            break

    print(f"\nFetch complete. Total valid records collected: {len(records)}")
    
    fieldnames = ['openalex_id', 'doi', 'title', 'publication_year', 'publication_date', 
                  'journal', 'abstract', 'authors', 'institutions', 'countries', 'concepts', 'cited_by_count']
    
    # 1. Save to CSV
    with open(output_csv, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(records)
    print(f"Saved CSV dataset to: {output_csv}")
    
    # 2. Save to JSON
    with open(output_json, 'w', encoding='utf-8') as f:
        json.dump(records, f, indent=2, ensure_ascii=False)
    print(f"Saved JSON dataset to: {output_json}")

    # 3. Save to Excel (.xlsx) using pandas
    df = pd.DataFrame(records)
    df.to_excel(output_excel, index=False)
    print(f"Saved Excel dataset to: {output_excel}")

if __name__ == "__main__":
    fetch_openalex_catalysis_abstracts(target_count=3000, from_year=2016)


Starting API fetch for 3000 records from OpenAlex...
Fetched 195 / 3000 valid records...
Fetched 385 / 3000 valid records...
Fetched 579 / 3000 valid records...
Fetched 771 / 3000 valid records...
Fetched 962 / 3000 valid records...
Fetched 1154 / 3000 valid records...
Fetched 1349 / 3000 valid records...
Fetched 1536 / 3000 valid records...
Fetched 1730 / 3000 valid records...
Fetched 1921 / 3000 valid records...
Fetched 2115 / 3000 valid records...
Fetched 2312 / 3000 valid records...
Fetched 2506 / 3000 valid records...
Fetched 2697 / 3000 valid records...
Fetched 2890 / 3000 valid records...
Fetched 3000 / 3000 valid records...

Fetch complete. Total valid records collected: 3000
Saved CSV dataset to: openalex_catalysts_3000.csv
Saved JSON dataset to: openalex_catalysts_3000.json
Saved Excel dataset to: openalex_catalysts_3000.xlsx
